In [7]:
# Notebook: sweep_inpaint_metrics.ipynb (single-cell version)
# Assumes:
#   data/images/1.png ... data/images/20.png
#   data/masks/1.png  ... data/masks/6.png
#
# Metrics (no GT needed):
#   - seam_grad_gap: |mean_grad_in - mean_grad_out|  on a boundary band
#   - seam_color_gap_lab: mean L2 color gap (Lab) between inner/outer boundary bands
#   - tv_band: total variation on the union band (lower is smoother)
#
# Runs:
#   baseline / repaint / bar_repaint with a param grid + multiple seeds
#
# Outputs:
#   outputs/sweep_<timestamp>/results.csv
#   outputs/sweep_<timestamp>/renders/<method>/<img>_<mask>/<seed>_<params>.png

import os, time, itertools, math
os.chdir("/workspace")
from dataclasses import asdict
from typing import Dict, Any, Tuple, List

import numpy as np
import pandas as pd
from PIL import Image
from tqdm.auto import tqdm

import torch
import torch.nn.functional as F

import cv2
from scipy.ndimage import distance_transform_edt, binary_dilation, binary_erosion

from diffusers import StableDiffusionInpaintPipeline


# ---- your callbacks ----
from src.samplers.latent_resample import (
    RePaintLikeConfig, BARConfig,
    build_callback_repaint_like, build_callback_bar_repaint
)

# ----------------------------
# Paths / I/O
# ----------------------------
IMAGES_DIR = "data/images"
MASKS_DIR  = "data/masks"
OUT_ROOT   = "outputs"
os.makedirs(OUT_ROOT, exist_ok=True)

run_tag = time.strftime("sweep_%Y-%m-%d_%H-%M-%S")
OUT_DIR = os.path.join(OUT_ROOT, run_tag)
RENDERS_DIR = os.path.join(OUT_DIR, "renders")
os.makedirs(RENDERS_DIR, exist_ok=True)

# ----------------------------
# Model / inference settings
# ----------------------------
MODEL_ID = "sd2-community/stable-diffusion-2-inpainting"  # or your cfg.model.hf_model_id
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DTYPE  = torch.float16  # try torch.bfloat16 if you want

# You can tune these:
PROMPT = "photorealistic, high quality"
NEG_PROMPT = ""
GUIDANCE = 7.5
NUM_STEPS = 30
WIDTH = 512
HEIGHT = 512

# Boundary band width in *image pixels*
BAND_PX = 5

# Seeds per setting (stability)
SEEDS = [0, 1, 2]

# ----------------------------
# Sweep grids
# ----------------------------
# Start small, then expand after you see trends.
BASELINE_GRID = [{}]

REPAINT_GRID = [
    {"jump_every": je, "p": p, "stop_jump_frac": sjf, "time_decay": True}
    for je in [7, 8, 9]
    for p  in [0.12, 0.15, 0.18]
    for sjf  in [0.6, 0.75]
]

BAR_GRID = [
    {"jump_every": je, "p_max": pm, "gamma": g, "rings": r, "stop_jump_frac": sjf, "time_decay": True}
    for je in [6, 7]
    for pm in [0.30, 0.35, 0.40]
    for g  in [2.0]
    for r  in [3]
    for sjf  in [0.6, 0.75]
]

METHODS = [
    ("baseline", BASELINE_GRID),
    ("repaint",  REPAINT_GRID),
    ("bar_repaint", BAR_GRID),
]

# ----------------------------
# Utils: load / resize
# ----------------------------
def load_image_rgb(path: str, size: Tuple[int,int]) -> Image.Image:
    img = Image.open(path).convert("RGB")
    return img.resize(size, resample=Image.BICUBIC)

def load_mask_l_binary(path: str, size: Tuple[int,int]) -> Image.Image:
    m = Image.open(path).convert("L")
    m = m.resize(size, resample=Image.NEAREST)
    arr = np.array(m, dtype=np.uint8)
    arr = (arr >= 128).astype(np.uint8) * 255  # white=fill
    return Image.fromarray(arr, mode="L")

def pil_to_np_rgb(img: Image.Image) -> np.ndarray:
    return np.array(img.convert("RGB"))

def pil_to_np_mask01(mask_l: Image.Image) -> np.ndarray:
    arr = np.array(mask_l, dtype=np.uint8)
    return (arr >= 128).astype(np.uint8)  # 1=fill

# ----------------------------
# Boundary band construction
# ----------------------------
def boundary_bands(mask01: np.ndarray, band_px: int) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
    """
    mask01: [H,W] 1=fill, 0=keep
    Returns:
      band_in  : inside-fill pixels within band_px of boundary
      band_out : outside-fill pixels within band_px of boundary
      band_union: band_in OR band_out
    """
    mask = mask01.astype(bool)
    keep = ~mask

    # distance to opposite set
    # inside band: fill pixels close to keep
    d_in  = distance_transform_edt(mask)   # distance inside fill to nearest keep
    d_out = distance_transform_edt(keep)   # distance outside fill to nearest fill

    band_in  = mask & (d_in  <= band_px) & (d_in > 0)
    band_out = keep & (d_out <= band_px) & (d_out > 0)
    band_union = band_in | band_out
    return band_in, band_out, band_union

# ----------------------------
# Metrics (no GT)
# ----------------------------
def sobel_grad_mag(gray01: np.ndarray) -> np.ndarray:
    gx = cv2.Sobel(gray01, cv2.CV_32F, 1, 0, ksize=3)
    gy = cv2.Sobel(gray01, cv2.CV_32F, 0, 1, ksize=3)
    return np.sqrt(gx*gx + gy*gy)

def total_variation(img_rgb01: np.ndarray, mask_bool: np.ndarray) -> float:
    """
    TV on masked area only; img_rgb01 float32 in [0,1], mask_bool [H,W]
    """
    # compute per-channel finite diffs
    dx = np.abs(img_rgb01[:, 1:, :] - img_rgb01[:, :-1, :])
    dy = np.abs(img_rgb01[1:, :, :] - img_rgb01[:-1, :, :])

    # align masks
    m_dx = mask_bool[:, 1:] & mask_bool[:, :-1]
    m_dy = mask_bool[1:, :] & mask_bool[:-1, :]

    tv = dx[m_dx].mean() if m_dx.any() else 0.0
    tv += dy[m_dy].mean() if m_dy.any() else 0.0
    return float(tv)

def seam_metrics(out_rgb: np.ndarray, mask01: np.ndarray, band_px: int) -> Dict[str, float]:
    """
    out_rgb: [H,W,3] uint8
    mask01: [H,W] {0,1}
    """
    band_in, band_out, band_union = boundary_bands(mask01, band_px=band_px)

    img01 = out_rgb.astype(np.float32) / 255.0
    gray = cv2.cvtColor(out_rgb, cv2.COLOR_RGB2GRAY).astype(np.float32) / 255.0
    grad = sobel_grad_mag(gray)

    # Seam gradient gap
    gin  = grad[band_in].mean()  if band_in.any()  else 0.0
    gout = grad[band_out].mean() if band_out.any() else 0.0
    seam_grad_gap = float(abs(gin - gout))

    # Color gap in Lab between bands (means)
    lab = cv2.cvtColor(out_rgb, cv2.COLOR_RGB2LAB).astype(np.float32)
    if band_in.any() and band_out.any():
        mu_in  = lab[band_in].mean(axis=0)
        mu_out = lab[band_out].mean(axis=0)
        seam_color_gap_lab = float(np.linalg.norm(mu_in - mu_out))
    else:
        seam_color_gap_lab = 0.0

    # Total variation on union band (smoothness proxy; lower is better)
    tv_band = total_variation(img01, band_union)

    # Also useful: edge density difference (optional)
    edge = (grad > 0.15).astype(np.uint8)  # threshold in [0,1]
    ein  = edge[band_in].mean()  if band_in.any()  else 0.0
    eout = edge[band_out].mean() if band_out.any() else 0.0
    edge_density_gap = float(abs(ein - eout))

    return dict(
        seam_grad_gap=seam_grad_gap,
        seam_color_gap_lab=seam_color_gap_lab,
        tv_band=tv_band,
        edge_density_gap=edge_density_gap,
        band_in_px=int(band_in.sum()),
        band_out_px=int(band_out.sum()),
    )

# ----------------------------
# Inference runner
# ----------------------------
def set_seed(seed: int):
    import random
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

pipe = StableDiffusionInpaintPipeline.from_pretrained(
    MODEL_ID,
    torch_dtype=DTYPE,
).to(DEVICE)
pipe.set_progress_bar_config(disable=True)

# Optional: compile for speed when doing many runs (enable when stable)
# pipe.unet = torch.compile(pipe.unet, mode="max-autotune", fullgraph=False)

def run_one(
    image_pil: Image.Image,
    mask_pil: Image.Image,
    method: str,
    params: Dict[str, Any],
    seed: int,
) -> Image.Image:
    set_seed(seed)

    callback = None
    if method == "repaint":
        rcfg = RePaintLikeConfig(
            jump_every=int(params["jump_every"]),
            p=float(params["p"]),
            stop_jump_frac=float(params.get("stop_jump_frac", 0.8)),
            time_decay=bool(params.get("time_decay", True)),
        )
        callback = build_callback_repaint_like(
            pipe=pipe, mask_l=mask_pil, cfg=rcfg,
            num_inference_steps=NUM_STEPS
        )

    elif method == "bar_repaint":
        rcfg = RePaintLikeConfig(
            jump_every=int(params["jump_every"]),
            p=0.0,
            stop_jump_frac=float(params.get("stop_jump_frac", 0.8)),
            time_decay=bool(params.get("time_decay", True)),
        )
        bcfg = BARConfig(
            p_max=float(params["p_max"]),
            gamma=float(params["gamma"]),
            rings=int(params["rings"]),
        )
        callback = build_callback_bar_repaint(
            pipe=pipe, mask_l=mask_pil,
            repaint_cfg=rcfg, bar_cfg=bcfg,
            num_inference_steps=NUM_STEPS
        )

    # baseline => callback None

    # AMP only on cuda + half
    use_amp = (DEVICE.type == "cuda" and DTYPE in (torch.float16, torch.bfloat16))
    amp_ctx = torch.autocast(device_type="cuda", dtype=DTYPE) if use_amp else torch.no_grad()

    with torch.inference_mode():
        if use_amp:
            ctx = torch.autocast(device_type="cuda", dtype=DTYPE)
        else:
            # dummy context
            class _Null:
                def __enter__(self): return None
                def __exit__(self, *a): return False
            ctx = _Null()

        with ctx:
            res = pipe(
                prompt=PROMPT,
                negative_prompt=NEG_PROMPT,
                image=image_pil,
                mask_image=mask_pil,
                guidance_scale=GUIDANCE,
                num_inference_steps=NUM_STEPS,
                callback_on_step_end=callback,
                callback_on_step_end_tensor_inputs=["latents"],
            )
    return res.images[0]

# ----------------------------
# Main sweep loop
# ----------------------------
# Image list: 1..20
image_ids = list(range(1, 21))
mask_ids  = list(range(1, 7))

rows = []

for img_id in tqdm(image_ids, desc="Images"):
    img_path = os.path.join(IMAGES_DIR, f"{img_id}.png")
    if not os.path.exists(img_path):
        print(f"[warn] missing image: {img_path}")
        continue

    image_pil = load_image_rgb(img_path, size=(WIDTH, HEIGHT))

    for m_id in mask_ids:
        mask_path = os.path.join(MASKS_DIR, f"{m_id}.png")
        if not os.path.exists(mask_path):
            print(f"[warn] missing mask: {mask_path}")
            continue

        mask_pil = load_mask_l_binary(mask_path, size=(WIDTH, HEIGHT))
        mask01 = pil_to_np_mask01(mask_pil)

        for method, grid in METHODS:
            for params in grid:
                # Render dir
                method_dir = os.path.join(RENDERS_DIR, method, f"img{img_id}_mask{m_id}")
                os.makedirs(method_dir, exist_ok=True)

                for seed in SEEDS:
                    out_pil = run_one(image_pil, mask_pil, method, params, seed)
                    out_np = pil_to_np_rgb(out_pil)

                    mets = seam_metrics(out_np, mask01, band_px=BAND_PX)

                    # Save image
                    # make compact filename
                    if method == "baseline":
                        fname = f"seed{seed}.png"
                    else:
                        ptag = "_".join([f"{k}{str(v).replace('.','p')}" for k,v in params.items()])
                        fname = f"seed{seed}__{ptag}.png"
                    out_file = os.path.join(method_dir, fname)
                    out_pil.save(out_file)

                    row = {
                        "img_id": img_id,
                        "mask_id": m_id,
                        "method": method,
                        "seed": seed,
                        "prompt": PROMPT,
                        "guidance": GUIDANCE,
                        "num_steps": NUM_STEPS,
                        "band_px": BAND_PX,
                        "out_file": out_file,
                        **(params if params else {}),
                        **mets,
                    }
                    rows.append(row)

df = pd.DataFrame(rows)

csv_path = os.path.join(OUT_DIR, "results.csv")
df.to_csv(csv_path, index=False)
print("Saved:", csv_path)

# Quick aggregate view: mean/std per method+params (helps pick optimal)
group_cols = ["method"]
# include params in grouping (only those present)
param_cols = sorted({c for c in df.columns if c in ["jump_every","p","p_max","gamma","rings","stop_jump_frac","time_decay"]})
group_cols += param_cols

agg = df.groupby(group_cols).agg(
    seam_grad_gap_mean=("seam_grad_gap","mean"),
    seam_grad_gap_std=("seam_grad_gap","std"),
    seam_color_gap_lab_mean=("seam_color_gap_lab","mean"),
    seam_color_gap_lab_std=("seam_color_gap_lab","std"),
    tv_band_mean=("tv_band","mean"),
    tv_band_std=("tv_band","std"),
).reset_index()

agg_path = os.path.join(OUT_DIR, "agg.csv")
agg.to_csv(agg_path, index=False)
print("Saved:", agg_path)

# Show top settings: prioritize low seam_grad_gap, then low color gap, then low tv
display(agg.sort_values(["seam_grad_gap_mean","seam_color_gap_lab_mean","tv_band_mean"]).head(20))


CLIPTextModel LOAD REPORT from: /workspace/.cache/huggingface/hub/models--sd2-community--stable-diffusion-2-inpainting/snapshots/5f74973cbb64c8568780732c17f43eb269d63a0d/text_encoder
Key                                | Status     |  | 
-----------------------------------+------------+--+-
text_model.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Saved: outputs/sweep_2026-02-22_14-18-04/results.csv
Saved: outputs/sweep_2026-02-22_14-18-04/agg.csv


,method,gamma,jump_every,p,p_max,rings,stop_jump_frac,time_decay,seam_grad_gap_mean,seam_grad_gap_std,seam_color_gap_lab_mean,seam_color_gap_lab_std,tv_band_mean,tv_band_std


In [3]:
import os, glob
print("CWD:", os.getcwd())
print("exists data/images:", os.path.exists("data/images"))
print("exists data/masks :", os.path.exists("data/masks"))
print("images sample:", glob.glob("data/images/*")[:10])
print("masks sample :", glob.glob("data/masks/*")[:10])


CWD: /workspace/notebooks
exists data/images: False
exists data/masks : False
images sample: []
masks sample : []


In [12]:
# Notebook cell: Fast LPIPS_full + LPIPS_masked (blend) over EXISTING renders
# - Uses results.csv (out_file, img_id, mask_id, ...)
# - Caches GT images + masks (huge speedup)
# - Computes LPIPS on GPU (if available)
# - Shows progress bar + ETA (tqdm)
# - Writes: results_lpips.csv + agg_lpips.csv into the same SWEEP_DIR

import os, glob, time
from functools import lru_cache

import numpy as np
import pandas as pd
from PIL import Image
from tqdm.auto import tqdm

import torch

# -----------------------
# 0) Paths
# -----------------------
os.chdir("/workspace")  # make relative paths consistent

# Set this explicitly OR leave None to auto-pick the latest sweep_*
SWEEP_DIR = "outputs/sweep_2026-02-23_11-09-43"  # e.g. "outputs/sweep_2026-02-19_07-53-55"

if SWEEP_DIR is None:
    cands = sorted(glob.glob("outputs/sweep_*"), key=os.path.getmtime)
    assert len(cands) > 0, "No outputs/sweep_* found"
    SWEEP_DIR = cands[-1]

RESULTS_CSV = os.path.join(SWEEP_DIR, "results.csv")
assert os.path.exists(RESULTS_CSV), f"Missing: {RESULTS_CSV}"

IMAGES_DIR = "data/images"
MASKS_DIR  = "data/masks"

print("Using SWEEP_DIR:", SWEEP_DIR)
print("Using RESULTS_CSV:", RESULTS_CSV)

# -----------------------
# 1) Device + LPIPS
# -----------------------
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("DEVICE:", DEVICE)
if DEVICE.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

try:
    import lpips  # pip install lpips
except ImportError as e:
    raise ImportError("Missing 'lpips'. Install: pip install lpips") from e

lpips_model = lpips.LPIPS(net="alex").to(DEVICE).eval()

def to_lpips_tensor(rgb_u8: np.ndarray) -> torch.Tensor:
    # [H,W,3] uint8 -> [1,3,H,W] float32 in [-1,1]
    t = torch.from_numpy(rgb_u8).permute(2,0,1).float() / 255.0
    t = t * 2.0 - 1.0
    return t.unsqueeze(0)

@torch.inference_mode()
def lpips_pair(pred_u8: np.ndarray, gt_u8: np.ndarray) -> float:
    pred_t = to_lpips_tensor(pred_u8).to(DEVICE, non_blocking=True)
    gt_t   = to_lpips_tensor(gt_u8).to(DEVICE, non_blocking=True)
    v = lpips_model(pred_t, gt_t)
    return float(v.item())

# -----------------------
# 2) Cached GT + mask loaders (resized to match output)
# -----------------------
@lru_cache(maxsize=256)
def load_gt_cached(img_id: int, w: int, h: int) -> np.ndarray:
    path = os.path.join(IMAGES_DIR, f"{img_id}.png")
    im = Image.open(path).convert("RGB").resize((w, h), resample=Image.BICUBIC)
    return np.array(im, dtype=np.uint8)

@lru_cache(maxsize=256)
def load_mask_cached(mask_id: int, w: int, h: int) -> np.ndarray:
    path = os.path.join(MASKS_DIR, f"{mask_id}.png")
    m = Image.open(path).convert("L").resize((w, h), resample=Image.NEAREST)
    arr = (np.array(m, dtype=np.uint8) >= 128).astype(np.uint8)  # 1=fill
    return arr

def load_pred(out_file: str) -> np.ndarray:
    # output image already at the sweep resolution
    im = Image.open(out_file).convert("RGB")
    return np.array(im, dtype=np.uint8)

# -----------------------
# 3) Compute LPIPS for all rows
#    LPIPS_masked uses "blend" trick:
#      blend = gt outside mask + pred inside mask
#      lpips_masked = LPIPS(blend, gt)
# -----------------------
df = pd.read_csv(RESULTS_CSV)
need_cols = {"out_file","img_id","mask_id"}
missing = need_cols - set(df.columns)
assert not missing, f"results.csv missing columns: {missing}"

lpips_full_vals = np.empty(len(df), dtype=np.float32)
lpips_masked_vals = np.empty(len(df), dtype=np.float32)
mask_area_fracs = np.empty(len(df), dtype=np.float32)

t0 = time.time()

# tqdm with rate/ETA
for idx, row in tqdm(df.iterrows(), total=len(df), desc="LPIPS"):
    out_file = row["out_file"]
    img_id = int(row["img_id"])
    mask_id = int(row["mask_id"])

    pred = load_pred(out_file)
    h, w = pred.shape[:2]

    gt = load_gt_cached(img_id, w, h)
    m01 = load_mask_cached(mask_id, w, h)  # [H,W] {0,1}

    # LPIPS full
    lp_full = lpips_pair(pred, gt)

    # LPIPS masked (blend)
    m = m01[..., None].astype(np.uint8)  # [H,W,1]
    blend = (gt * (1 - m) + pred * m).astype(np.uint8)
    lp_masked = lpips_pair(blend, gt)

    lpips_full_vals[idx] = lp_full
    lpips_masked_vals[idx] = lp_masked
    mask_area_fracs[idx] = float(m01.mean())

df["lpips_full"] = lpips_full_vals
df["lpips_masked"] = lpips_masked_vals
df["mask_area_frac"] = mask_area_fracs

elapsed = time.time() - t0
print(f"Done in {elapsed/60:.1f} minutes for {len(df)} rows")

# -----------------------
# 4) Save per-run + aggregate
# -----------------------
out_results = os.path.join(SWEEP_DIR, "results_lpips.csv")
df.to_csv(out_results, index=False)
print("Saved:", out_results)

# Aggregate like before
param_cols = ["jump_every","p","p_max","gamma","rings","stop_jump_frac","time_decay"]
for c in param_cols:
    if c in df.columns:
        df[c] = df[c].fillna("NA")

group_cols = ["method"] + [c for c in param_cols if c in df.columns]

agg = (
    df.groupby(group_cols)
      .agg(
          lpips_full_mean=("lpips_full","mean"),
          lpips_full_std=("lpips_full","std"),
          lpips_masked_mean=("lpips_masked","mean"),
          lpips_masked_std=("lpips_masked","std"),
          seam_grad_gap_mean=("seam_grad_gap","mean") if "seam_grad_gap" in df.columns else ("lpips_full","mean"),
          seam_color_gap_lab_mean=("seam_color_gap_lab","mean") if "seam_color_gap_lab" in df.columns else ("lpips_full","mean"),
          tv_band_mean=("tv_band","mean") if "tv_band" in df.columns else ("lpips_full","mean"),
      )
      .reset_index()
)

out_agg = os.path.join(SWEEP_DIR, "agg_lpips.csv")
agg.to_csv(out_agg, index=False)
print("Saved:", out_agg)

# Show best configs by LPIPS_masked first (then seam if present)
sort_cols = ["lpips_masked_mean", "lpips_full_mean"]
if "seam_grad_gap_mean" in agg.columns:
    sort_cols.insert(1, "seam_grad_gap_mean")
display(agg.sort_values(sort_cols).head(20))

Using SWEEP_DIR: outputs/sweep_2026-02-23_11-09-43
Using RESULTS_CSV: outputs/sweep_2026-02-23_11-09-43/results.csv
DEVICE: cuda
GPU: NVIDIA GeForce RTX 5090 Laptop GPU
Setting up [LPIPS] perceptual loss: trunk [alex], v[0.1], spatial [off]


/opt/micromamba/envs/repaint-bar/lib/python3.11/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/opt/micromamba/envs/repaint-bar/lib/python3.11/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=AlexNet_Weights.IMAGENET1K_V1`. You can also use `weights=AlexNet_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Loading model from: /opt/micromamba/envs/repaint-bar/lib/python3.11/site-packages/lpips/weights/v0.1/alex.pth
Done in 9.6 minutes for 36000 rows
Saved: outputs/sweep_2026-02-23_11-09-43/results_lpips.csv
Saved: outputs/sweep_2026-02-23_11-09-43/agg_lpips.csv


,method,jump_every,p,p_max,gamma,rings,stop_jump_frac,time_decay,lpips_full_mean,lpips_full_std,lpips_masked_mean,lpips_masked_std,seam_grad_gap_mean,seam_color_gap_lab_mean,tv_band_mean
62,repaint,4.0,0.13,NA,NA,NA,0.4,True,0.150004,0.072688,0.084291,0.040181,0.013527,2.541064,0.055009
63,repaint,4.0,0.15,NA,NA,NA,0.533,True,0.150075,0.073140,0.084452,0.040529,0.013512,2.538511,0.054738
6,bar_repaint,4.0,NA,0.278,1.008,1.0,0.4,True,0.150507,0.072594,0.084714,0.040332,0.015114,2.549926,0.056106
64,repaint,5.0,0.093,NA,NA,NA,0.5,True,0.150899,0.071869,0.084957,0.039830,0.015285,2.562215,0.056031
8,bar_repaint,4.0,NA,0.319,2.238,3.0,0.533,True,0.151098,0.072113,0.085139,0.040179,0.016668,2.576379,0.056677
9,bar_repaint,4.0,NA,0.502,2.057,3.0,0.533,True,0.150980,0.072858,0.085168,0.040667,0.015653,2.577715,0.056365
11,bar_repaint,4.0,NA,0.544,2.366,3.0,0.533,True,0.151041,0.072756,0.085199,0.040635,0.015946,2.583129,0.056469
17,bar_repaint,5.0,NA,0.489,1.532,3.0,0.5,True,0.151071,0.072322,0.085217,0.040072,0.015994,2.584150,0.056491
1,bar_repaint,4.0,NA,0.217,1.779,1.0,0.667,True,0.151279,0.072212,0.085355,0.040281,0.016572,2.584534,0.056673
13,bar_repaint,5.0,NA,0.212,1.185,1.0,0.667,True,0.151328,0.071926,0.085356,0.039982,0.016625,2.578864,0.056662


In [5]:
import pandas as pd, os
df = pd.read_csv("outputs/sweep_2026-02-19_07-53-55/results.csv")
print("rows:", len(df))
print("first out_file:", df.loc[0,"out_file"])
print("exists?", os.path.exists(df.loc[0,"out_file"]))

rows: 24120
first out_file: outputs/sweep_2026-02-19_07-53-55/renders/baseline/img1_mask1/seed0.png
exists? True


In [ ]:
code with prompts:

In [11]:
# Notebook: sweep_inpaint_metrics.ipynb (single-cell version) — UPDATED (budgeted random search)
# Changes applied:
#   ✅ Per-image prompt dict stays
#   ✅ Higher guidance stays
#   ✅ Replace huge grids with RANDOM search under a fixed budget
#   ✅ BAR grid no longer explodes (sample N configs)
#   ✅ Prints estimated total renders before running
#
# Still saves renders + seam metrics (no GT). LPIPS can be computed later from renders.

import os, time, math
os.chdir("/workspace")
from typing import Dict, Any, Tuple, List

import numpy as np
import pandas as pd
from PIL import Image
from tqdm.auto import tqdm

import torch
import cv2
from scipy.ndimage import distance_transform_edt

from diffusers import StableDiffusionInpaintPipeline

# ---- your callbacks ----
from src.samplers.latent_resample import (
    RePaintLikeConfig, BARConfig,
    build_callback_repaint_like, build_callback_bar_repaint
)

# ----------------------------
# Paths / I/O
# ----------------------------
IMAGES_DIR = "data/images"
MASKS_DIR  = "data/masks"
OUT_ROOT   = "outputs"
os.makedirs(OUT_ROOT, exist_ok=True)

run_tag = time.strftime("sweep_%Y-%m-%d_%H-%M-%S")
OUT_DIR = os.path.join(OUT_ROOT, run_tag)
RENDERS_DIR = os.path.join(OUT_DIR, "renders")
os.makedirs(RENDERS_DIR, exist_ok=True)

# ----------------------------
# Model / inference settings
# ----------------------------
MODEL_ID = "sd2-community/stable-diffusion-2-inpainting"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DTYPE  = torch.float16

DEFAULT_PROMPT = "high quality"
DEFAULT_NEG_PROMPT = "cartoon, painting, oversaturated, unrealistic, blurry, artifacts, deformed, extra objects, watermark, text"
PROMPT_SUFFIX = "inpaint the masked region only, matching the existing lighting and background, same perspective, consistent shadows, photorealistic"

GUIDANCE = 7.5
NUM_STEPS = 30
WIDTH = 512
HEIGHT = 512

BAND_PX = 5
SEEDS = [0, 1, 2]

# ----------------------------
# Per-image prompts
# ----------------------------
PROMPTS_BY_IMAGE: Dict[int, str] = {
    1: "Orange sun, partially behind clouds",
    2: "Night city lights with winding illuminated road from a hilltop view",
    3: "Children riding bikes on a sunny dirt road",
    4: "Colorful ocean sunset with dramatic clouds over a beach",
    5: "Cluster of tall palm trees with dense green foliage",
    6: "Silhouetted trees against a pink and purple sunset sky",
    7: "Stack of pancakes with syrup and a scoop of chocolate ice cream",
    8: "Hand holding a bunch of dark purple grapes",
    9: "Wooded park with stone steps and leafy trees",
    10: "Bouquet of pink lilies with green buds",
    11: "Close-up of a red rose in bloom",
    12: "Purple wildflower growing among green vegetation and rocks",
    13: "Plate with breaded cutlet, pasta shells, green beans, and onion rings",
    14: "Underground parking garage with parked cars and yellow P2 signs",
    15: "Bee on yellow and white blossoms",
    16: "Toddler walking on grassy hill beside a tree",
    17: "City street with a green bus and traffic",
    18: "Three colorful tulips in a small vase with eucalyptus leaves",
    19: "Wooden bench in front of a stone wall and white picket fence",
    20: "Three children on a grassy lawn near trees and cacti",
}

NEG_PROMPTS_BY_IMAGE: Dict[int, str] = {}

def get_prompt(img_id: int) -> str:
    return f"{PROMPTS_BY_IMAGE.get(img_id, DEFAULT_PROMPT)}, {PROMPT_SUFFIX}"

def get_neg_prompt(img_id: int) -> str:
    return NEG_PROMPTS_BY_IMAGE.get(img_id, DEFAULT_NEG_PROMPT)

# ----------------------------
# Budgeted random search configs
# ----------------------------
# Set your experiment budget here (configs, not renders)
# With 20 images * 6 masks * 3 seeds = 360 renders per config.
N_REPAINT = 40
N_BAR     = 60
RNG_SEED  = 1234

rng = np.random.default_rng(RNG_SEED)

def discrete_sjf_candidates(num_steps: int, jump_every: int) -> List[float]:
    """
    Produce discrete stop_jump_frac candidates that actually change behavior.

    We construct candidates around the scheduler jump steps:
      jump steps are: s where (s+1) % jump_every == 0
    stop_step = int(sjf * num_steps). If stop_step passes a jump step, that jump is dropped.
    So we place sjf thresholds just before/after those jump steps.
    """
    jumps = [s for s in range(num_steps) if (s + 1) % jump_every == 0]
    # stop_step is integer in [0..num_steps]
    # condition to allow jump at step s: s < stop_step
    # So boundary at stop_step = s+1
    stop_steps = sorted(set([0, num_steps] + [s + 1 for s in jumps]))

    # Convert stop_steps into sjf via sjf = stop_step / num_steps
    # Keep within sensible range (avoid trivial always-on / always-off unless you want)
    cands = []
    for st in stop_steps:
        sjf = st / float(num_steps)
        cands.append(sjf)

    # Optional: prune extremes (you can tweak)
    # keep only values that roughly correspond to "stop in second half"
    cands = [x for x in cands if 0.4 <= x <= 0.95]

    # Dedup + sort + round
    cands = sorted({round(x, 3) for x in cands})
    return cands

def sample_repaint(n: int) -> List[Dict[str, Any]]:
    out = []
    for _ in range(n):
        je = int(rng.integers(4, 11))                # 4..10
        p  = float(rng.uniform(0.08, 0.30))          # 0.08..0.30
        sjf_cands = discrete_sjf_candidates(NUM_STEPS, je)
        sjf = float(rng.choice(sjf_cands)) if sjf_cands else 0.8
        out.append({
            "jump_every": je,
            "p": round(p, 3),
            "stop_jump_frac": round(sjf, 3),
            "time_decay": True,
        })
    # de-dup
    uniq = {tuple(sorted(d.items())): d for d in out}
    return list(uniq.values())

def sample_bar(n: int) -> List[Dict[str, Any]]:
    out = []
    rings_choices = [1, 3, 5]
    for _ in range(n):
        je   = int(rng.integers(4, 10))              # 4..9
        pmax = float(rng.uniform(0.20, 0.60))        # 0.20..0.60
        gamma = float(rng.uniform(1.0, 3.0))         # 1.0..3.0
        rings = int(rng.choice(rings_choices))
        sjf_cands = discrete_sjf_candidates(NUM_STEPS, je)
        sjf = float(rng.choice(sjf_cands)) if sjf_cands else 0.8
        out.append({
            "jump_every": je,
            "p_max": round(pmax, 3),
            "gamma": round(gamma, 3),
            "rings": rings,
            "stop_jump_frac": round(sjf, 3),
            "time_decay": True,
        })
    uniq = {tuple(sorted(d.items())): d for d in out}
    return list(uniq.values())

BASELINE_GRID = [{}]
REPAINT_GRID = sample_repaint(N_REPAINT)
BAR_GRID = sample_bar(N_BAR)

METHODS = [
    ("baseline", BASELINE_GRID),
    ("repaint",  REPAINT_GRID),
    ("bar_repaint", BAR_GRID),
]

print("Total configs:")
print(" baseline:", len(BASELINE_GRID))
print(" repaint :", len(REPAINT_GRID))
print(" bar     :", len(BAR_GRID))

# Estimate total renders
n_img = 20
n_mask = 6
n_seed = len(SEEDS)
n_cfg = len(BASELINE_GRID) + len(REPAINT_GRID) + len(BAR_GRID)
total_renders = n_img * n_mask * n_seed * n_cfg
print("Total renders (approx):", total_renders, f"(~{n_img*n_mask*n_seed} per config)")

# ----------------------------
# Utils: load / resize
# ----------------------------
def load_image_rgb(path: str, size: Tuple[int,int]) -> Image.Image:
    img = Image.open(path).convert("RGB")
    return img.resize(size, resample=Image.BICUBIC)

def load_mask_l_binary(path: str, size: Tuple[int,int]) -> Image.Image:
    m = Image.open(path).convert("L")
    m = m.resize(size, resample=Image.NEAREST)
    arr = np.array(m, dtype=np.uint8)
    arr = (arr >= 128).astype(np.uint8) * 255
    return Image.fromarray(arr, mode="L")

def pil_to_np_rgb(img: Image.Image) -> np.ndarray:
    return np.array(img.convert("RGB"))

def pil_to_np_mask01(mask_l: Image.Image) -> np.ndarray:
    arr = np.array(mask_l, dtype=np.uint8)
    return (arr >= 128).astype(np.uint8)

# ----------------------------
# Boundary band construction
# ----------------------------
def boundary_bands(mask01: np.ndarray, band_px: int):
    mask = mask01.astype(bool)
    keep = ~mask
    d_in  = distance_transform_edt(mask)
    d_out = distance_transform_edt(keep)
    band_in  = mask & (d_in  <= band_px) & (d_in > 0)
    band_out = keep & (d_out <= band_px) & (d_out > 0)
    band_union = band_in | band_out
    return band_in, band_out, band_union

# ----------------------------
# Metrics (no GT)
# ----------------------------
def sobel_grad_mag(gray01: np.ndarray) -> np.ndarray:
    gx = cv2.Sobel(gray01, cv2.CV_32F, 1, 0, ksize=3)
    gy = cv2.Sobel(gray01, cv2.CV_32F, 0, 1, ksize=3)
    return np.sqrt(gx*gx + gy*gy)

def total_variation(img_rgb01: np.ndarray, mask_bool: np.ndarray) -> float:
    dx = np.abs(img_rgb01[:, 1:, :] - img_rgb01[:, :-1, :])
    dy = np.abs(img_rgb01[1:, :, :] - img_rgb01[:-1, :, :])
    m_dx = mask_bool[:, 1:] & mask_bool[:, :-1]
    m_dy = mask_bool[1:, :] & mask_bool[:-1, :]
    tv = dx[m_dx].mean() if m_dx.any() else 0.0
    tv += dy[m_dy].mean() if m_dy.any() else 0.0
    return float(tv)

def seam_metrics(out_rgb: np.ndarray, mask01: np.ndarray, band_px: int) -> Dict[str, float]:
    band_in, band_out, band_union = boundary_bands(mask01, band_px=band_px)

    img01 = out_rgb.astype(np.float32) / 255.0
    gray = cv2.cvtColor(out_rgb, cv2.COLOR_RGB2GRAY).astype(np.float32) / 255.0
    grad = sobel_grad_mag(gray)

    gin  = grad[band_in].mean()  if band_in.any()  else 0.0
    gout = grad[band_out].mean() if band_out.any() else 0.0
    seam_grad_gap = float(abs(gin - gout))

    lab = cv2.cvtColor(out_rgb, cv2.COLOR_RGB2LAB).astype(np.float32)
    if band_in.any() and band_out.any():
        mu_in  = lab[band_in].mean(axis=0)
        mu_out = lab[band_out].mean(axis=0)
        seam_color_gap_lab = float(np.linalg.norm(mu_in - mu_out))
    else:
        seam_color_gap_lab = 0.0

    tv_band = total_variation(img01, band_union)

    edge = (grad > 0.15).astype(np.uint8)
    ein  = edge[band_in].mean()  if band_in.any()  else 0.0
    eout = edge[band_out].mean() if band_out.any() else 0.0
    edge_density_gap = float(abs(ein - eout))

    return dict(
        seam_grad_gap=seam_grad_gap,
        seam_color_gap_lab=seam_color_gap_lab,
        tv_band=tv_band,
        edge_density_gap=edge_density_gap,
        band_in_px=int(band_in.sum()),
        band_out_px=int(band_out.sum()),
    )

# ----------------------------
# Inference runner
# ----------------------------
def set_seed(seed: int):
    import random
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

pipe = StableDiffusionInpaintPipeline.from_pretrained(
    MODEL_ID,
    torch_dtype=DTYPE,
).to(DEVICE)
pipe.set_progress_bar_config(disable=True)

# Optional: compile for speed when doing many runs
# pipe.unet = torch.compile(pipe.unet, mode="max-autotune", fullgraph=False)

def run_one(
    image_pil: Image.Image,
    mask_pil: Image.Image,
    prompt: str,
    neg_prompt: str,
    method: str,
    params: Dict[str, Any],
    seed: int,
) -> Image.Image:
    set_seed(seed)

    callback = None
    if method == "repaint":
        rcfg = RePaintLikeConfig(
            jump_every=int(params["jump_every"]),
            p=float(params["p"]),
            stop_jump_frac=float(params.get("stop_jump_frac", 0.8)),
            time_decay=bool(params.get("time_decay", True)),
        )
        callback = build_callback_repaint_like(
            pipe=pipe, mask_l=mask_pil, cfg=rcfg, num_inference_steps=NUM_STEPS
        )

    elif method == "bar_repaint":
        rcfg = RePaintLikeConfig(
            jump_every=int(params["jump_every"]),
            p=0.0,
            stop_jump_frac=float(params.get("stop_jump_frac", 0.8)),
            time_decay=bool(params.get("time_decay", True)),
        )
        bcfg = BARConfig(
            p_max=float(params["p_max"]),
            gamma=float(params["gamma"]),
            rings=int(params["rings"]),
        )
        callback = build_callback_bar_repaint(
            pipe=pipe, mask_l=mask_pil, repaint_cfg=rcfg, bar_cfg=bcfg,
            num_inference_steps=NUM_STEPS
        )

    use_amp = (DEVICE.type == "cuda" and DTYPE in (torch.float16, torch.bfloat16))

    with torch.inference_mode():
        if use_amp:
            ctx = torch.autocast(device_type="cuda", dtype=DTYPE)
        else:
            class _Null:
                def __enter__(self): return None
                def __exit__(self, *a): return False
            ctx = _Null()

        with ctx:
            gen = torch.Generator(device="cuda").manual_seed(int(seed)) if DEVICE.type=="cuda" else None
            res = pipe(
                prompt=prompt,
                negative_prompt=neg_prompt,
                image=image_pil,
                mask_image=mask_pil,
                guidance_scale=float(GUIDANCE),
                num_inference_steps=int(NUM_STEPS),
                callback_on_step_end=callback,
                callback_on_step_end_tensor_inputs=["latents"],
                generator=gen,
            )
    return res.images[0]

# ----------------------------
# Main sweep loop
# ----------------------------
image_ids = list(range(1, 21))
mask_ids  = list(range(1, 7))

rows = []

total = len(image_ids) * len(mask_ids) * (len(BASELINE_GRID)+len(REPAINT_GRID)+len(BAR_GRID)) * len(SEEDS)
pbar = tqdm(total=total, desc="Sweep", leave=True)

for img_id in image_ids:
    img_path = os.path.join(IMAGES_DIR, f"{img_id}.png")
    if not os.path.exists(img_path):
        print(f"[warn] missing image: {img_path}")
        pbar.update(len(mask_ids) * (len(BASELINE_GRID)+len(REPAINT_GRID)+len(BAR_GRID)) * len(SEEDS))
        continue

    image_pil = load_image_rgb(img_path, size=(WIDTH, HEIGHT))
    prompt = get_prompt(img_id)
    neg_prompt = get_neg_prompt(img_id)

    for m_id in mask_ids:
        mask_path = os.path.join(MASKS_DIR, f"{m_id}.png")
        if not os.path.exists(mask_path):
            print(f"[warn] missing mask: {mask_path}")
            pbar.update((len(BASELINE_GRID)+len(REPAINT_GRID)+len(BAR_GRID)) * len(SEEDS))
            continue

        mask_pil = load_mask_l_binary(mask_path, size=(WIDTH, HEIGHT))
        mask01 = pil_to_np_mask01(mask_pil)

        for method, grid in METHODS:
            for params in grid:
                method_dir = os.path.join(RENDERS_DIR, method, f"img{img_id}_mask{m_id}")
                os.makedirs(method_dir, exist_ok=True)

                for seed in SEEDS:
                    out_pil = run_one(image_pil, mask_pil, prompt, neg_prompt, method, params, seed)
                    out_np = pil_to_np_rgb(out_pil)

                    mets = seam_metrics(out_np, mask01, band_px=BAND_PX)

                    if method == "baseline":
                        fname = f"seed{seed}.png"
                    else:
                        ptag = "_".join([f"{k}{str(v).replace('.','p')}" for k,v in params.items()])
                        fname = f"seed{seed}__{ptag}.png"

                    out_file = os.path.join(method_dir, fname)
                    out_pil.save(out_file)

                    row = {
                        "img_id": img_id,
                        "mask_id": m_id,
                        "method": method,
                        "seed": seed,
                        "prompt": prompt,
                        "negative_prompt": neg_prompt,
                        "guidance": GUIDANCE,
                        "num_steps": NUM_STEPS,
                        "band_px": BAND_PX,
                        "out_file": out_file,
                        **(params if params else {}),
                        **mets,
                    }
                    rows.append(row)
                    pbar.update(1)

pbar.close()

df = pd.DataFrame(rows)
csv_path = os.path.join(OUT_DIR, "results.csv")
df.to_csv(csv_path, index=False)
print("Saved:", csv_path)

param_cols = sorted({c for c in df.columns if c in ["jump_every","p","p_max","gamma","rings","stop_jump_frac","time_decay"]})
group_cols = ["method"] + param_cols

agg = df.groupby(group_cols).agg(
    seam_grad_gap_mean=("seam_grad_gap","mean"),
    seam_grad_gap_std=("seam_grad_gap","std"),
    seam_color_gap_lab_mean=("seam_color_gap_lab","mean"),
    seam_color_gap_lab_std=("seam_color_gap_lab","std"),
    tv_band_mean=("tv_band","mean"),
    tv_band_std=("tv_band","std"),
).reset_index()

agg_path = os.path.join(OUT_DIR, "agg.csv")
agg.to_csv(agg_path, index=False)
print("Saved:", agg_path)

display(agg.sort_values(["seam_grad_gap_mean","seam_color_gap_lab_mean","tv_band_mean"]).head(20))

Total configs:
 baseline: 1
 repaint : 39
 bar     : 60
Total renders (approx): 36000 (~360 per config)


CLIPTextModel LOAD REPORT from: /workspace/.cache/huggingface/hub/models--sd2-community--stable-diffusion-2-inpainting/snapshots/5f74973cbb64c8568780732c17f43eb269d63a0d/text_encoder
Key                                | Status     |  | 
-----------------------------------+------------+--+-
text_model.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Saved: outputs/sweep_2026-02-23_11-09-43/results.csv
Saved: outputs/sweep_2026-02-23_11-09-43/agg.csv


,method,gamma,jump_every,p,p_max,rings,stop_jump_frac,time_decay,seam_grad_gap_mean,seam_grad_gap_std,seam_color_gap_lab_mean,seam_color_gap_lab_std,tv_band_mean,tv_band_std


In [ ]:
recreate agg.csv:

In [13]:
import os, re, math, time
from pathlib import Path
from typing import Dict, Any, Tuple, List

import numpy as np
import pandas as pd
from PIL import Image
from tqdm.auto import tqdm

import torch
import torch.nn.functional as F

# Optional (for LAB on GPU). If not installed, we fallback to OpenCV CPU.
try:
    import kornia
    _HAVE_KORNIA = True
except Exception:
    _HAVE_KORNIA = False

try:
    import cv2
    _HAVE_CV2 = True
except Exception:
    _HAVE_CV2 = False


# =========================
# CONFIG
# =========================
SWEEP_DIR = "outputs/sweep_2026-02-23_11-09-43"   # <<< CHANGE ME
IMAGES_DIR = "data/images"
MASKS_DIR  = "data/masks"

WIDTH = 512
HEIGHT = 512
BAND_PX = 5

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("DEVICE:", DEVICE)

RENDERS_DIR = Path(SWEEP_DIR) / "renders"
OUT_AGG = Path(SWEEP_DIR) / "agg.csv"   # rebuild exactly this
OUT_RESULTS_REBUILT = Path(SWEEP_DIR) / "results_rebuilt_from_renders.csv"  # optional debug


# =========================
# Helpers: mask + boundary bands (computed ONCE per mask_id)
# =========================
def load_mask01(mask_path: str, size=(WIDTH, HEIGHT)) -> np.ndarray:
    m = Image.open(mask_path).convert("L")
    m = m.resize(size, resample=Image.NEAREST)
    arr = np.array(m, dtype=np.uint8)
    return (arr >= 128).astype(np.uint8)  # 1=fill

def boundary_bands_from_mask01(mask01: np.ndarray, band_px: int) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
    """
    CPU precompute of bands. Only depends on mask, so do it once per mask id.
    Uses distance transform (CPU), but only for 6 masks -> fast.
    """
    from scipy.ndimage import distance_transform_edt
    mask = mask01.astype(bool)
    keep = ~mask
    d_in  = distance_transform_edt(mask)
    d_out = distance_transform_edt(keep)
    band_in  = mask & (d_in  <= band_px) & (d_in > 0)
    band_out = keep & (d_out <= band_px) & (d_out > 0)
    band_union = band_in | band_out
    return band_in, band_out, band_union

def np_bool_to_torch(mask_bool: np.ndarray, device: torch.device) -> torch.Tensor:
    # [H,W] bool -> [1,1,H,W] float/bool
    t = torch.from_numpy(mask_bool.astype(np.bool_))[None, None, ...].to(device)
    return t


# =========================
# Helpers: parse metadata from path
# renders/<method>/img{img}_mask{mask}/seed{seed}__kV_...png
# =========================
PARAM_KEYS = ["jump_every","p","p_max","gamma","rings","stop_jump_frac","time_decay"]

def decode_val(s: str):
    # values have "." replaced by "p" (e.g., 0p15), ints are "3", bool "True"
    if s in ("True","False"):
        return (s == "True")
    # numeric
    ss = s.replace("p", ".")
    try:
        if "." in ss:
            return float(ss)
        return int(ss)
    except Exception:
        # fallback: string
        return s

def parse_file_info(p: Path) -> Dict[str, Any]:
    # method from parent dirs
    # .../renders/<method>/img{img}_mask{mask}/<fname>.png
    method = p.parents[1].name
    img_mask = p.parent.name  # "imgX_maskY"
    m = re.match(r"img(\d+)_mask(\d+)", img_mask)
    if not m:
        raise ValueError(f"Bad folder name: {img_mask}")
    img_id = int(m.group(1))
    mask_id = int(m.group(2))

    fname = p.stem  # without .png
    # baseline: seed0
    # others: seed0__jump_every8_p0p15_...
    if "__" in fname:
        left, right = fname.split("__", 1)
    else:
        left, right = fname, ""

    seed_m = re.match(r"seed(\d+)", left)
    if not seed_m:
        raise ValueError(f"Bad filename seed: {fname}")
    seed = int(seed_m.group(1))

    params = {}
    if right:
        parts = right.split("_")
        # token like "jump_every8" or "p0p15"
        for tok in parts:
            for k in PARAM_KEYS:
                if tok.startswith(k):
                    v = tok[len(k):]
                    params[k] = decode_val(v)
                    break

    return dict(img_id=img_id, mask_id=mask_id, method=method, seed=seed, **params)


# =========================
# GPU metrics (Sobel/TV/edge gap) + LAB color gap
# =========================
_sobel_x = torch.tensor([[-1,0,1],[-2,0,2],[-1,0,1]], dtype=torch.float32)[None,None,:,:]
_sobel_y = torch.tensor([[-1,-2,-1],[0,0,0],[1,2,1]], dtype=torch.float32)[None,None,:,:]

def pil_to_rgb_tensor(pil_img: Image.Image, device: torch.device) -> torch.Tensor:
    # uint8 [H,W,3] -> float32 [1,3,H,W] in [0,1]
    arr = np.array(pil_img.convert("RGB"), dtype=np.uint8)
    t = torch.from_numpy(arr).to(device=device, dtype=torch.float32) / 255.0
    t = t.permute(2,0,1).unsqueeze(0)  # [1,3,H,W]
    return t

def sobel_grad_mag(gray01: torch.Tensor) -> torch.Tensor:
    # gray01: [1,1,H,W] float
    gx = F.conv2d(gray01, _sobel_x.to(gray01.device), padding=1)
    gy = F.conv2d(gray01, _sobel_y.to(gray01.device), padding=1)
    return torch.sqrt(gx*gx + gy*gy + 1e-12)  # [1,1,H,W]

def total_variation_on_band(img01: torch.Tensor, band_union: torch.Tensor) -> float:
    # img01: [1,3,H,W], band_union: [1,1,H,W] bool
    # dx: [1,3,H,W-1], dy: [1,3,H-1,W]
    dx = (img01[:,:,:,1:] - img01[:,:,:,:-1]).abs()
    dy = (img01[:,:,1:,:] - img01[:,:,:-1,:]).abs()

    bu = band_union.bool()
    m_dx = bu[:,:,:,1:] & bu[:,:,:,:-1]
    m_dy = bu[:,:,1:,:] & bu[:,:,:-1,:]

    tv = 0.0
    if m_dx.any():
        tv += dx[m_dx.expand_as(dx)].mean().item()
    if m_dy.any():
        tv += dy[m_dy.expand_as(dy)].mean().item()
    return float(tv)

def seam_color_gap_lab(out_rgb_uint8: np.ndarray, band_in: np.ndarray, band_out: np.ndarray) -> float:
    # Best-effort: GPU with kornia if available, else CPU with cv2
    if not band_in.any() or not band_out.any():
        return 0.0

    if _HAVE_KORNIA:
        # GPU path
        t = torch.from_numpy(out_rgb_uint8).to(DEVICE, dtype=torch.float32) / 255.0  # [H,W,3]
        t = t.permute(2,0,1).unsqueeze(0)  # [1,3,H,W]
        lab = kornia.color.rgb_to_lab(t).squeeze(0).permute(1,2,0)  # [H,W,3]
        bin_t  = torch.from_numpy(band_in).to(DEVICE)
        bout_t = torch.from_numpy(band_out).to(DEVICE)
        mu_in  = lab[bin_t].mean(dim=0)
        mu_out = lab[bout_t].mean(dim=0)
        return float(torch.linalg.norm(mu_in - mu_out).item())

    # CPU path (fast enough; IO dominates anyway)
    if not _HAVE_CV2:
        # fallback: RGB gap (not LAB)
        mu_in = out_rgb_uint8[band_in].mean(axis=0)
        mu_out = out_rgb_uint8[band_out].mean(axis=0)
        return float(np.linalg.norm(mu_in - mu_out))

    lab = cv2.cvtColor(out_rgb_uint8, cv2.COLOR_RGB2LAB).astype(np.float32)
    mu_in  = lab[band_in].mean(axis=0)
    mu_out = lab[band_out].mean(axis=0)
    return float(np.linalg.norm(mu_in - mu_out))


def compute_metrics_for_image(
    img_pil: Image.Image,
    band_in_np: np.ndarray,
    band_out_np: np.ndarray,
    band_union_np: np.ndarray,
    band_in_t: torch.Tensor,
    band_out_t: torch.Tensor,
    band_union_t: torch.Tensor,
    edge_thresh: float = 0.15,
) -> Dict[str, float]:
    """
    Compute seam_grad_gap, edge_density_gap, tv_band on GPU;
    seam_color_gap_lab via GPU(kornia) or CPU(cv2).
    """
    img01 = pil_to_rgb_tensor(img_pil, DEVICE)               # [1,3,H,W]
    gray01 = (0.2989*img01[:,0:1] + 0.5870*img01[:,1:2] + 0.1140*img01[:,2:3])  # [1,1,H,W]

    grad = sobel_grad_mag(gray01)  # [1,1,H,W]

    # means
    bi = band_in_t.bool()
    bo = band_out_t.bool()

    gin  = grad[bi].mean().item()  if bi.any() else 0.0
    gout = grad[bo].mean().item()  if bo.any() else 0.0
    seam_grad_gap = float(abs(gin - gout))

    edge = (grad > edge_thresh)
    ein  = edge[bi].float().mean().item() if bi.any() else 0.0
    eout = edge[bo].float().mean().item() if bo.any() else 0.0
    edge_density_gap = float(abs(ein - eout))

    tv_band = total_variation_on_band(img01, band_union_t)

    # LAB color gap (best-effort)
    out_rgb_uint8 = (img01.squeeze(0).permute(1,2,0).clamp(0,1).mul(255).byte().detach().cpu().numpy())
    seam_color_gap = seam_color_gap_lab(out_rgb_uint8, band_in_np, band_out_np)

    return dict(
        seam_grad_gap=seam_grad_gap,
        seam_color_gap_lab=seam_color_gap,
        tv_band=tv_band,
        edge_density_gap=edge_density_gap,
        band_in_px=int(band_in_np.sum()),
        band_out_px=int(band_out_np.sum()),
    )


# =========================
# Build mask bands cache
# =========================
mask_cache = {}
for mask_id in range(1, 7):
    mp = Path(MASKS_DIR) / f"{mask_id}.png"
    if not mp.exists():
        print("[warn] missing mask:", mp)
        continue
    m01 = load_mask01(str(mp))
    bi, bo, bu = boundary_bands_from_mask01(m01, BAND_PX)
    mask_cache[mask_id] = dict(
        band_in_np=bi, band_out_np=bo, band_union_np=bu,
        band_in_t=np_bool_to_torch(bi, DEVICE),
        band_out_t=np_bool_to_torch(bo, DEVICE),
        band_union_t=np_bool_to_torch(bu, DEVICE),
    )

print("Cached masks:", sorted(mask_cache.keys()))


# =========================
# Scan renders and compute
# =========================
pngs = sorted(RENDERS_DIR.rglob("*.png"))
if not pngs:
    raise FileNotFoundError(f"No PNGs found under {RENDERS_DIR}")

rows = []
t0 = time.time()

for p in tqdm(pngs, desc="Recomputing metrics from renders"):
    info = parse_file_info(p)

    mid = info["mask_id"]
    if mid not in mask_cache:
        continue

    try:
        img_pil = Image.open(p).convert("RGB")
    except Exception as e:
        print("[warn] failed to read", p, e)
        continue

    c = mask_cache[mid]
    mets = compute_metrics_for_image(
        img_pil,
        c["band_in_np"], c["band_out_np"], c["band_union_np"],
        c["band_in_t"],  c["band_out_t"],  c["band_union_t"],
        edge_thresh=0.15
    )

    row = dict(
        out_file=str(p),
        band_px=BAND_PX,
        **info,
        **mets
    )
    rows.append(row)

dt = time.time() - t0
print(f"Done computing per-render metrics: {len(rows)} rows in {dt/60:.2f} minutes")

df = pd.DataFrame(rows)

# Save rebuilt per-render results (optional but useful)
df.to_csv(OUT_RESULTS_REBUILT, index=False)
print("Saved:", OUT_RESULTS_REBUILT)

# =========================
# Aggregate -> agg.csv
# =========================
param_cols = [c for c in PARAM_KEYS if c in df.columns]
group_cols = ["method"] + param_cols

agg = (
    df.groupby(group_cols, dropna=False)
      .agg(
          seam_grad_gap_mean=("seam_grad_gap","mean"),
          seam_grad_gap_std=("seam_grad_gap","std"),
          seam_color_gap_lab_mean=("seam_color_gap_lab","mean"),
          seam_color_gap_lab_std=("seam_color_gap_lab","std"),
          tv_band_mean=("tv_band","mean"),
          tv_band_std=("tv_band","std"),
          edge_density_gap_mean=("edge_density_gap","mean"),
          edge_density_gap_std=("edge_density_gap","std"),
          n=("seam_grad_gap","count"),
      )
      .reset_index()
)

agg.to_csv(OUT_AGG, index=False)
print("Saved rebuilt agg.csv:", OUT_AGG)

display(agg.sort_values(["seam_grad_gap_mean","seam_color_gap_lab_mean","tv_band_mean"]).head(20))

DEVICE: cuda
Cached masks: [1, 2, 3, 4, 5, 6]
Done computing per-render metrics: 36000 rows in 8.71 minutes
Saved: outputs/sweep_2026-02-23_11-09-43/results_rebuilt_from_renders.csv
Saved rebuilt agg.csv: outputs/sweep_2026-02-23_11-09-43/agg.csv


,method,p,gamma,rings,seam_grad_gap_mean,seam_grad_gap_std,seam_color_gap_lab_mean,seam_color_gap_lab_std,tv_band_mean,tv_band_std,edge_density_gap_mean,edge_density_gap_std,n
72,repaint,0.15,NaN,NaN,0.013515,0.013257,2.538511,2.341779,0.054738,0.025275,0.022259,0.018830,360
68,repaint,0.13,NaN,NaN,0.013520,0.013583,2.541064,2.354972,0.055009,0.025387,0.022354,0.019228,360
67,repaint,0.129,NaN,NaN,0.013760,0.013043,2.558832,2.340620,0.054244,0.024914,0.022372,0.018820,360
79,repaint,0.214,NaN,NaN,0.013909,0.013499,2.589414,2.441588,0.054473,0.024976,0.022692,0.019497,360
92,repaint,0.288,NaN,NaN,0.014409,0.015076,2.635632,2.488183,0.055301,0.025282,0.023613,0.019636,360
95,repaint,0.297,NaN,NaN,0.014824,0.013456,2.653553,2.453059,0.053731,0.024550,0.023188,0.019490,360
80,repaint,0.215,NaN,NaN,0.014864,0.015555,2.620870,2.479153,0.055696,0.025471,0.024343,0.020213,360
1,bar_repaint,,1.008,1.0,0.015058,0.015189,2.549926,2.378798,0.056106,0.025778,0.023527,0.020024,360
0,bar_repaint,,1.000,3.0,0.015290,0.015612,2.595568,2.438567,0.056148,0.025674,0.023582,0.020715,360
7,bar_repaint,,1.123,3.0,0.015290,0.014800,2.600539,2.385656,0.055406,0.025221,0.023129,0.019999,360
